<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z334_BreakDetector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detector de Quiebre Estructural + Fallback

## La intuición

Si una serie sufrió un cambio de nivel abrupto **que ya es visible en los datos** (el quiebre ocurrió antes del período de corte), entonces ningún modelo entrenado sobre la historia larga va a predecir bien — porque sigue anclado al nivel viejo.

La solución no es un modelo más sofisticado: es **olvidar la historia** y anclarse al nuevo nivel.

```
Historia larga  ████████████████░░░░░░   ← nivel viejo
                                 ▼ quiebre
Últimos meses                    ▓▓▓▓▓   ← nuevo nivel

HAR / AutoGluon: predice en el nivel viejo  → error grande
Fallback:        predice media últimos 3m   → error chico
```

## Detector

```
z_break = (media_últimos_3m - media_histórica) / std_histórico

si |z_break| > umbral  →  QUIEBRE → fallback = media últimos 3m
si no                  →  NORMAL  → modelo elegido (HAR / reg6m)
```

## Backtesting
Train hasta 201910, target 201912 — mismo split que z330.

## 0. Init Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "product_id_apredecir201912.txt"

# 1. Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle

In [ ]:
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from scipy.stats import chi2, runstest_1samp
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings('ignore')

In [ ]:
PARAM = {
    'experimento':        'BreakDetector-01',
    'kaggle_competition': 'labo-iii-2026-rosario',

    # detector de quiebre
    'ventana_reciente':   3,    # meses para el "nuevo nivel"
    'ventana_historia':   12,   # meses de historia para calcular la media larga
    'umbral_z':           2.0,  # sigmas para declarar quiebre

    # modelo base (cuando NO hay quiebre)
    'ventana_reg':        6,    # ventana para reg lineal reciente

    # backtesting
    'periodo_corte':      201910,
    'periodo_target':     201912,
}

ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2. Datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")
tb_ventas    = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])

tb_train = tb_ventas.filter(pl.col("periodo") <= PARAM['periodo_corte'])
tb_real  = (
    tb_ventas
    .filter(pl.col("periodo") == PARAM['periodo_target'])
    .select(["product_id", "tn"])
    .rename({"tn": "tn_real"})
)

productos = tb_apredecir["product_id"].to_list()
print(f"{len(productos)} productos")

# 3. Detector de quiebre estructural

Compara el nivel reciente vs la historia larga en términos de sigmas.

```
z_break = (media_últimos_k - media_histórica) / std_histórica
```

La historia excluye los últimos k meses para no contaminar la referencia con el nivel nuevo.

In [ ]:
def detectar_quiebre(serie: np.ndarray, ventana_reciente: int,
                     ventana_historia: int, umbral_z: float):
    """
    Devuelve (es_quiebre: bool, z_break: float, nivel_nuevo: float)

    - ventana_reciente: meses para estimar el 'nuevo nivel'
    - ventana_historia: meses previos a lo reciente para estimar el nivel base
    - umbral_z: sigmas para declarar quiebre
    """
    n = len(serie)
    k = min(ventana_reciente, n)
    h = min(ventana_historia, n - k)

    if h < 3:
        return False, 0.0, float(serie[-k:].mean())

    reciente  = serie[-k:]
    historia  = serie[-(k + h):-k]

    mu_hist   = historia.mean()
    std_hist  = historia.std() + 1e-9
    mu_rec    = reciente.mean()

    z = (mu_rec - mu_hist) / std_hist
    es_quiebre = abs(z) > umbral_z

    return es_quiebre, float(z), float(mu_rec)


def pred_reg_lineal(serie: np.ndarray, ventana: int, horizonte: int = 2) -> float:
    w = min(ventana, len(serie))
    if w < 2:
        return max(float(serie.mean()), 0.0)
    y = serie[-w:]
    x = np.arange(w).reshape(-1, 1)
    pred = float(LinearRegression().fit(x, y).predict([[w - 1 + horizonte]])[0])
    return max(pred, 0.0)


def pred_har(serie: np.ndarray) -> float:
    if len(serie) < 14:
        return max(float(serie.mean()), 0.0)
    try:
        _, pv = runstest_1samp(serie, cutoff='median')
        if pv >= 0.05:
            return max(float(serie[-12:].mean()), 0.0)
    except Exception:
        pass

    X, y = [], []
    for t in range(12, len(serie)):
        X.append([serie[t-1], serie[t-3:t].mean(), serie[t-6:t].mean(), serie[t-12:t].mean()])
        y.append(serie[t])
    X, y = np.array(X), np.array(y)

    m = LinearRegression().fit(X, y)

    def one_step(s):
        t = len(s)
        x = [[s[t-1], s[t-3:t].mean(), s[t-6:t].mean(), s[t-12:t].mean()]]
        return max(float(m.predict(x)[0]), 0.0)

    p1 = one_step(serie)
    p2 = one_step(np.append(serie, p1))
    return p2

# 4. Backtesting — todos los modelos

In [ ]:
vr  = PARAM['ventana_reciente']
vh  = PARAM['ventana_historia']
uz  = PARAM['umbral_z']
vreg = PARAM['ventana_reg']

resultados = []

for pid in productos:
    serie = (
        tb_train.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    # naive: mediana últimos 6m
    naive = max(float(np.median(serie[-6:])), 0.0)

    # reg lineal reciente (sin detector)
    reg6m = pred_reg_lineal(serie, ventana=vreg, horizonte=2)

    # HAR (sin detector)
    har = pred_har(serie)

    # detector de quiebre
    es_quiebre, z_break, nivel_nuevo = detectar_quiebre(serie, vr, vh, uz)

    # fallback: media de los últimos ventana_reciente meses (el nuevo nivel)
    fallback = max(nivel_nuevo, 0.0)

    # modelo híbrido: quiebre → fallback, si no → reg6m
    break_reg6m = fallback if es_quiebre else reg6m

    # modelo híbrido: quiebre → fallback, si no → HAR
    break_har   = fallback if es_quiebre else har

    resultados.append({
        'product_id':   pid,
        'es_quiebre':   es_quiebre,
        'z_break':      z_break,
        'pred_naive':   naive,
        'pred_reg6m':   reg6m,
        'pred_har':     har,
        'pred_fallback': fallback,
        'pred_break_reg6m': break_reg6m,
        'pred_break_har':   break_har,
    })

tb_preds = pl.DataFrame(resultados)
n_quiebre = tb_preds['es_quiebre'].sum()
print(f"Productos con quiebre detectado (|z| > {uz}): {n_quiebre} de {len(productos)} ({100*n_quiebre/len(productos):.1f}%)")

In [ ]:
tb_bt = tb_real.join(tb_preds, on='product_id', how='left')

modelos = ['naive', 'reg6m', 'har', 'fallback', 'break_reg6m', 'break_har']
for m in modelos:
    tb_bt = tb_bt.with_columns(
        (pl.col('tn_real') - pl.col(f'pred_{m}')).abs().alias(f'err_{m}')
    )

print(f"RMSE — backtesting 201912  (umbral_z={uz}, ventana_reciente={vr}m)")
print()
rmse_naive = float(np.sqrt((tb_bt['err_naive'] ** 2).mean()))
for m in modelos:
    rmse = float(np.sqrt((tb_bt[f'err_{m}'] ** 2).mean()))
    delta = rmse - rmse_naive
    tag = '(baseline)' if m == 'naive' else f'({delta:+.4f} vs naive)'
    print(f"  {m:18s}: {rmse:.4f}  {tag}")

# 5. ¿El detector es correcto?

Comparamos el error de `fallback` vs `reg6m` **solo en los productos donde se detectó quiebre**.
Si el detector funciona, fallback debería ganar claramente en ese subconjunto.

In [ ]:
tb_con_quiebre = tb_bt.filter(pl.col('es_quiebre') == True)
tb_sin_quiebre = tb_bt.filter(pl.col('es_quiebre') == False)

print(f"Productos CON quiebre detectado: {tb_con_quiebre.height}")
print()
for m in ['naive', 'reg6m', 'har', 'fallback']:
    rmse = float(np.sqrt((tb_con_quiebre[f'err_{m}'] ** 2).mean()))
    print(f"  {m:12s}: {rmse:.4f}")

print()
print(f"Productos SIN quiebre detectado: {tb_sin_quiebre.height}")
print()
for m in ['naive', 'reg6m', 'har', 'fallback']:
    rmse = float(np.sqrt((tb_sin_quiebre[f'err_{m}'] ** 2).mean()))
    print(f"  {m:12s}: {rmse:.4f}")

# 6. Sensibilidad al umbral

¿Qué pasa con el RMSE global si cambiamos el umbral entre 1 y 3 sigmas?
Buscamos el punto donde el detector agrega más valor.

In [ ]:
umbrales = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0]

# precalculamos z_break por producto (ya lo tenemos en tb_preds)
z_breaks    = tb_bt['z_break'].to_numpy()
err_reg6m   = tb_bt['err_reg6m'].to_numpy()
err_fallback= tb_bt['err_fallback'].to_numpy()
tn_real_sq  = tb_bt['tn_real'].to_numpy()

print(f"{'umbral':>8}  {'n_quiebre':>10}  {'RMSE break_reg6m':>18}  {'RMSE reg6m':>12}")
print("-" * 58)

rmse_results = []
for uz_ in umbrales:
    mask = np.abs(z_breaks) > uz_
    n_q  = mask.sum()
    # break_reg6m: fallback si quiebre, reg6m si no
    err_hibrido = np.where(mask, err_fallback, err_reg6m)
    rmse_ = float(np.sqrt((err_hibrido ** 2).mean()))
    rmse_results.append((uz_, n_q, rmse_))
    print(f"  {uz_:6.1f}  {n_q:10d}  {rmse_:18.4f}")

rmse_base_reg6m = float(np.sqrt((err_reg6m ** 2).mean()))
print(f"\n  {'reg6m puro':>8}  {'':>10}  {rmse_base_reg6m:18.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

uz_list   = [r[0] for r in rmse_results]
nq_list   = [r[1] for r in rmse_results]
rmse_list = [r[2] for r in rmse_results]

axes[0].plot(uz_list, rmse_list, 'o-', color='steelblue', linewidth=2)
axes[0].axhline(rmse_base_reg6m, color='tomato', linestyle='--', label='reg6m puro')
axes[0].set_xlabel('umbral z')
axes[0].set_ylabel('RMSE global')
axes[0].set_title('RMSE vs umbral de detección de quiebre')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(uz_list, nq_list, color='steelblue', alpha=0.7, width=0.3)
axes[1].set_xlabel('umbral z')
axes[1].set_ylabel('productos detectados como quiebre')
axes[1].set_title('N° productos con quiebre según umbral')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 7. Visualización — casos donde el detector ayuda y donde falla

Buscamos los productos donde `break_reg6m` ganó más al `reg6m` (detector útil)
y los que el detector empeoró (falsos positivos).

In [ ]:
tb_bt = tb_bt.with_columns(
    (pl.col('err_reg6m') - pl.col('err_break_reg6m')).alias('mejora_detector')
)

# top 6 donde el detector ayudó (solo en los que se activó)
top_aciertos = (
    tb_bt.filter(pl.col('es_quiebre') == True)
    .sort('mejora_detector', descending=True)
    .head(6)['product_id'].to_list()
)

# top 6 donde el detector empeoró (falsos positivos)
top_errores = (
    tb_bt.filter(pl.col('es_quiebre') == True)
    .sort('mejora_detector')
    .head(6)['product_id'].to_list()
)

def plot_casos(pids, titulo):
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes = axes.flatten()

    for i, pid in enumerate(pids):
        serie_full = tb_ventas.filter(pl.col('product_id') == pid).sort('periodo')
        periodos_  = serie_full['periodo'].to_list()
        tn_        = serie_full['tn'].to_numpy().astype(float)

        idx_corte  = next((j for j, p in enumerate(periodos_) if p > PARAM['periodo_corte']), len(periodos_))
        idx_target = next((j for j, p in enumerate(periodos_) if p == PARAM['periodo_target']), None)
        tn_train_  = tn_[:idx_corte]

        row       = tb_bt.filter(pl.col('product_id') == pid)
        real_val  = float(row['tn_real'][0])
        reg6_val  = float(row['pred_reg6m'][0])
        fall_val  = float(row['pred_fallback'][0])
        har_val   = float(row['pred_har'][0])
        z_val     = float(row['z_break'][0])
        mejora    = float(row['mejora_detector'][0])

        ax = axes[i]

        # historia — tiñe los últimos ventana_reciente meses de otro color
        n_rec = min(PARAM['ventana_reciente'], len(tn_train_))
        ax.plot(range(len(tn_train_) - n_rec), tn_train_[:-n_rec],
                'o-', color='steelblue', markersize=3, linewidth=1.5, label='historia')
        ax.plot(range(len(tn_train_) - n_rec, len(tn_train_)), tn_train_[-n_rec:],
                'o-', color='darkorange', markersize=4, linewidth=2, label=f'últimos {n_rec}m (nuevo nivel)')

        # línea horizontal del nivel nuevo (fallback)
        ax.axhline(fall_val, color='darkorange', linestyle=':', linewidth=1.2, alpha=0.7)

        if idx_target is not None:
            t = idx_target
            ax.scatter([t], [real_val], color='black',     s=90, zorder=6, label=f'real={real_val:.1f}')
            ax.scatter([t], [fall_val], color='darkorange',s=60, zorder=5, marker='D', label=f'fallback={fall_val:.1f}')
            ax.scatter([t], [reg6_val], color='tomato',    s=60, zorder=5, marker='D', label=f'reg6m={reg6_val:.1f}')
            ax.scatter([t], [har_val],  color='green',     s=40, zorder=5, marker='D', label=f'HAR={har_val:.1f}')

        ax.set_title(f'pid {pid}  |  z={z_val:.2f}  mejora={mejora:.1f}', fontsize=8)
        ax.legend(fontsize=5)

    fig.suptitle(titulo, fontsize=10)
    plt.tight_layout()
    plt.show()

plot_casos(top_aciertos, 'Detector ACERTÓ — fallback ganó a reg6m')
plot_casos(top_errores,  'Detector FALLÓ — fallback empeoró vs reg6m (falsos positivos)')

# 8. McNemar — significancia

In [ ]:
def mcnemar(err_a, err_b, nombre_a, nombre_b):
    n10 = (err_a < err_b).sum()
    n01 = (err_b < err_a).sum()
    if n10 + n01 == 0:
        print(f"{nombre_a} vs {nombre_b}: sin discrepancias")
        return
    chi2_stat = (abs(n10 - n01) - 1)**2 / (n10 + n01)
    pvalue    = 1 - chi2.cdf(chi2_stat, df=1)
    ganador   = nombre_a if n10 > n01 else nombre_b
    sig = "** SIG **" if pvalue < 0.05 else "no sig   "
    print(f"{nombre_a:18s} vs {nombre_b:18s}:  gana {n10:3d}|{n01:3d}  p={pvalue:.4f}  {sig}  → {ganador}")

print("McNemar global (780 productos):")
print()
err_naive   = tb_bt['err_naive'].to_numpy()
err_reg6m   = tb_bt['err_reg6m'].to_numpy()
err_har     = tb_bt['err_har'].to_numpy()
err_br6     = tb_bt['err_break_reg6m'].to_numpy()
err_bhar    = tb_bt['err_break_har'].to_numpy()

mcnemar(err_naive, err_reg6m,  'naive',       'reg6m')
mcnemar(err_naive, err_har,    'naive',       'har')
mcnemar(err_naive, err_br6,    'naive',       'break_reg6m')
mcnemar(err_naive, err_bhar,   'naive',       'break_har')
print()
mcnemar(err_reg6m, err_br6,    'reg6m',       'break_reg6m')
mcnemar(err_har,   err_bhar,   'har',         'break_har')
mcnemar(err_br6,   err_bhar,   'break_reg6m', 'break_har')

# 9. Submit — predicción 202002

Usa toda la historia hasta 201912 para predecir 202002 (t+2).
Cambiá `modelo_base` en PARAM_SUBMIT para probar las variantes.

In [ ]:
PARAM_SUBMIT = {
    'modelo_base': 'reg6m',  # 'reg6m' o 'har'
    'ventana_reciente': PARAM['ventana_reciente'],
    'ventana_historia': PARAM['ventana_historia'],
    'umbral_z':         PARAM['umbral_z'],
    'ventana_reg':      PARAM['ventana_reg'],
}

tb_full     = tb_ventas
preds_final = []

for pid in productos:
    serie = (
        tb_full.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    es_quiebre, z_break, nivel_nuevo = detectar_quiebre(
        serie,
        PARAM_SUBMIT['ventana_reciente'],
        PARAM_SUBMIT['ventana_historia'],
        PARAM_SUBMIT['umbral_z'],
    )

    if es_quiebre:
        pred = max(nivel_nuevo, 0.0)
    else:
        if PARAM_SUBMIT['modelo_base'] == 'reg6m':
            pred = pred_reg_lineal(serie, ventana=PARAM_SUBMIT['ventana_reg'], horizonte=2)
        else:
            pred = pred_har(serie)

    preds_final.append({'product_id': pid, 'tn': pred})

tb_final = pl.DataFrame(preds_final)
display(tb_final.head(10))
print(f"Nulls: {tb_final['tn'].is_null().sum()}")

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
    os.system(f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"')

mb   = PARAM_SUBMIT['modelo_base']
uz_  = PARAM_SUBMIT['umbral_z']
vr_  = PARAM_SUBMIT['ventana_reciente']

archivo = f"BreakDetector_{mb}_z{uz_}_rec{vr_}m.csv"
mensaje = f"Break detector z>{uz_} reciente={vr_}m → fallback, else {mb}"

tb_final.write_csv(archivo)
kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)